# Mercor Cheating Detection
## I. Introduction
### A. Problem Description
Our goal in this challenge is to predict whether an candidate is engaging in cheating behavior during an interview. We have data that includes anonymized behavioral features, platform activity signals, and a social graph that captures relationships between users across the network.  
### B. Evaluation
Evaluations are performed using a cost-based metric that is designed to reflect real-world operational costs used in cheating detection. Our evaluation should automatically search for optimal decision thresholds that minimize total cost across the following decision regions:
1. Auto-pass (low cheating risk)
2. Manual review (medium cheating risk)
3. Auto-block (high cheating risk)  

The cost structure for this evaluation is defined as follows:
- False Negative (cheating passes through): $600
- False Positive in auto-block region: $300
- False Positive in manual review region: $150
- True Positive requiring manual review: $5
- Correct auto-pass or auto-block: $0  
The score is the negative of the minimum total cost found across all threshold cominations. Higher is better. Our model will need to make confident, correct decisions while penalizing costly operational errors.

## II. Setup
### A. Imports
We are importing standard ML workflow libraries. We'll stick to proven models for this baseline evaluation.

In [2]:
from pathlib import Path 

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier 
from matplotlib.axes import Axes 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from tqdm import tqdm

### B. Configuration

In [ ]:
class CFG:
    # Data configuration
    DATA_DIR = ("../input/mercor-cheating-detection")
    TRAIN_PATH = DATA_DIR / Path("train.csv")
    TEST_PATH = DATA_DIR / Path("test.csv")
    SOCIAL_GRAPH_PATH = DATA_DIR / Path("social_graph.csv")
    SAMPLE_SUB_PATH = DATA_DIR / Path("sample_submission.csv")

    # Hyperparameter configuration
    SEED = 15
    VALID_SIZE = 0.1

    # Other configuration
    NUMERIC_FEATURES = [
        "feature_001", "feature_002", "feature_003", "feature_004", "feature_005",
        "feature_006", "feature_008", "feature_009", "feature_010", "feature_012",
        "feature_015", "feature_016", "feature_017", "feature_018"
    ]
    BINARY_FEATURES = [
        "feature_007", "feature_011", "feature_013", "feature_014", "high_conf_clean"
    ]

### C. Load and Preview Data

In [9]:
train = pd.read_csv(CFG.TRAIN_PATH)
social_graph = pd.read_csv(CFG.SOCIAL_GRAPH_PATH)
test = pd.read_csv(CFG.TEST_PATH)
sample = pd.read_csv(CFG.SAMPLE_SUB_PATH)

print("*" * 80)
print(f"Train shape: {train.shape}")
print(train.info(), "\n")
print("*" * 80)
print(f"Social graph shape: {social_graph.shape}")
print(social_graph.info(), "\n")
print("*" * 80)
print(f"Test shape: {test.shape}")
print(test.info(), "\n")
print("*" * 80)
print(f"Sample shape: {sample.shape}")
print(sample.info(), "\n")

********************************************************************************
Train shape: (272819, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 272819 entries, 0 to 272818
Data columns (total 21 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   user_hash        272819 non-null  object 
 1   feature_001      240704 non-null  float64
 2   feature_002      241114 non-null  float64
 3   feature_003      241114 non-null  float64
 4   feature_004      270420 non-null  float64
 5   feature_005      267020 non-null  float64
 6   feature_006      270605 non-null  float64
 7   feature_007      266984 non-null  float64
 8   feature_008      266723 non-null  float64
 9   feature_009      266723 non-null  float64
 10  feature_010      266723 non-null  float64
 11  feature_011      267003 non-null  float64
 12  feature_012      261527 non-null  float64
 13  feature_013      267003 non-null  float64
 14  feature_014      266939 n

In [13]:
print("Features and their unique value counts:")
print(train.nunique())

Features and their unique value counts:
user_hash          272819
feature_001             5
feature_002            10
feature_003            10
feature_004            11
feature_005            10
feature_006            11
feature_007             2
feature_008            11
feature_009            11
feature_010          8154
feature_011             2
feature_012             4
feature_013             2
feature_014             2
feature_015        155430
feature_016           177
feature_017         71025
feature_018         22638
high_conf_clean         1
is_cheating             2
dtype: int64


#### Initial Insights & Hypotheses
Based on the metadata, unique value counts, and missingness patterns, we can draw the following inferences:

**Disjoint Labeling Structure:** There are **272,819** total rows. The sum of non-null values in `high_conf_clean` (112,966) and `is_cheating` (159,853) exactly equals the total. *Hypothesis:* The dataset is perfectly disjoint. `NaN`s in `is_cheating` are likely structural zeros representing the "clean" class.


**"Hidden" Categorical/Ordinal Features:**
Despite being listed as `NUMERIC_FEATURES`, several columns have extremely low cardinality:
* **`feature_012`** has only **4** unique values.
* **`feature_001`** has only **5** unique values.
* **`feature_002`–`006`, `008`, `009**` have **10–11** unique values.
* *Hypothesis:* These are likely **Ordinal Ranks** (1st to 10th) or **Encoded Nominal Categories** (Type 1, Type 2, etc.), not continuous measurements. Treating them as purely continuous (e.g., standardizing/scaling) may mask their discrete nature.


**Structural Missingness (Groups):**
* **Group A (`002`, `003`):** 11.7% missing. Both are low-cardinality numeric (10 unique values).
* **Group B (`008`, `009`, `010`):** 2.7% missing. Mixed cardinality (`010` is high-cardinality continuous; `008`/`009` are low-cardinality).
* **Group C (`011`, `012`, `013`):** 2.6% missing. This is a highly mixed group: Binary (`011`, `013`) + Low-Cardinality Numeric (`012` with only 4 values).



#### Actions & Future Workflow

##### A. Immediate Validation (Baseline Scope)

1. **Verify Disjoint Assumption:** Assert that no row has both `high_conf_clean` and `is_cheating` populated.
2. **Impute Target Variable:** Fill `NaN` values in `is_cheating` with `0`.
3. **Inspect Low-Cardinality Numerics:** For `feature_001` (5 vals) and `feature_012` (4 vals), print the `.value_counts()`.
* *Goal:* Determine if the values are sequential (e.g., 1, 2, 3, 4 -> Ordinal) or arbitrary (e.g., 10, 50, 99 -> Nominal codes).
4. **Metadata Alignment:** Ensure `BINARY_FEATURES` strictly contain {0, 1}.

##### B. Advanced Strategies (Post-Baseline Scope)
5. **Encoding Experiments:** For the low-cardinality "numerics" (`001`, `012`), test **One-Hot Encoding** vs. keeping them as raw numbers. If they are nominal categories labeled 1-5, keeping them as numbers implies a mathematical relationship (5 > 1) that may not exist.
6. **Ordinal Encodings:** For the "Rank-like" features (10–11 unique values like `002`–`009`), test tree-based models (which handle these natively) vs. linear models (which may require OHE or Target Encoding).
7. **Informative Missingness (Group C):** Since `feature_012` is in Group C and has only 4 values, check if the *missingness* in Group C correlates with specific values in other features.
8. **Stratified Validation:** Ensure the validation split respects the ratio of Manual vs. High-Confidence labels.

In [19]:
def validate_label_assumptions(df):
    """
    Validates the assumption that 'is_cheating' (manual labels) and 
    'high_conf_clean' (auto-labels) are mutually exclusive.
    """
    overlap_mask = (df['high_conf_clean'] == 1) & (df['is_cheating'].notna())
    num_overlap = overlap_mask.sum()
    missing_mask = (df['high_conf_clean'].isna()) & (df['is_cheating'].isna())
    num_missing = missing_mask.sum()
    num_manual = df['is_cheating'].notna().sum()
    num_auto = df['high_conf_clean'].sum()
    total_rows = len(df)

    print(f"Total Rows: {total_rows}")
    print(f"Manual Labels (is_cheating): {num_manual}")
    print(f"Auto Labels (high_conf_clean): {num_auto}")
    
    if num_overlap == 0:
        print("The sets are strictly disjoint.")
    else:
        print(f"Found {num_overlap} overlapping rows.")
        conflicts = df[overlap_mask]
        print(conflicts['is_cheating'].value_counts())
    if num_missing == 0:
        print("Every row has either a manual label or high_conf flag.")
    else:
        print(f"Found {num_missing} rows with nO information.")
    return num_overlap == 0

validate_label_assumptions(train)

Total Rows: 272819
Manual Labels (is_cheating): 112966
Auto Labels (high_conf_clean): 159853.0
The sets are strictly disjoint.
Every row has either a manual label or high_conf flag.


np.True_